# Imports

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from statsmodels.tsa.seasonal import STL
from plotnine import *
from mizani.formatters import date_format

from hypertrees.models import HyperTreeSTL
from experiments.utils import (
    load_experiments_specs,
    install_legend_justification_backport,
)

install_legend_justification_backport()

repo_root = Path.cwd()
while not (repo_root / 'pyproject.toml').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

seed = 123
figure_size = (14, 10)
plots_dir = repo_root / 'experiments' / 'runs' / 'results' / 'plots'
plots_dir.mkdir(parents=True, exist_ok=True)

def save_plot(fcst_plots):
    for p in fcst_plots:
        yield p + theme(figure_size=figure_size)

# Data

In [ ]:
experiment_specs = load_experiments_specs(
    dataset="airpassengers",
    train_type="local",
)

# Train and Test Data
train_df = experiment_specs["train"]
train_df["year_feat_ts"] = train_df["date"].dt.year

old_feat_names = ["time_idx", "month_feat_ts", "quarter_feat_ts", "year_feat_ts"]
new_feat_names = ["time", "month", "quarter", "year"]
feature_rename_dict = {old: new for old, new in zip(old_feat_names, new_feat_names)}
train_df.rename(columns=feature_rename_dict, inplace=True)

test_df = experiment_specs["test"]
test_df["year_feat_ts"] = test_df["date"].dt.year
test_df.rename(columns=feature_rename_dict, inplace=True)

features = new_feat_names
col_sel = ["series_id", "date", "value"] + features

# Train model

In [ ]:
np.random.seed(seed)

ht_stl = HyperTreeSTL(
    period=12,
    num_seasonal_components=1,
    freq='M',
    type="paper",
)

# Train model
ht_stl.train(
    lgb_params={
        'learning_rate': 0.3,
        'random_seed': 123,
        'verbose': -1,
    },
    num_iterations=500,
    train_data=train_df[col_sel],
)

# Extract components and parameters

In [ ]:
# Extract trend and seasonality
stl_df = ht_stl.forecast(
    train_df,
    type="components",
)
trend = stl_df["trend"].to_numpy().reshape(-1, 1)
seasonality = stl_df["seasonality"].to_numpy().reshape(-1, 1)

# Extract STL parameters
model_params = ht_stl.forecast(
    train_df,
    type="parameters",
).drop(columns=["series_id", "model"])

# Classical STL Decomposition

In [ ]:
# Classical
y_train = train_df["value"].to_numpy().reshape(-1)
airp_stl = pd.Series(
    y_train, index=train_df["date"], name="airp"
)

stl = STL(airp_stl, seasonal=13)
res = stl.fit()

# Plots

### Trend

In [ ]:
color_map = {"STL": "#1f77b4", "Hyper-Tree": "#FFA500"}
size_map = {"Hyper-Tree": 1.8, "STL": 1.8}

trend_df_ht = pd.DataFrame({
    "date": train_df["date"],
    "value": trend.reshape(-1,),
    "type": "Hyper-Tree",
})

trend_stl = pd.DataFrame({
    "date": train_df["date"],
    "value": res._trend.values,
    "type": "STL",
})

trend_df = pd.concat([trend_df_ht, trend_stl], axis=0)

trend_plot = []
trend_plot.append(
    ggplot(trend_df,
           aes(x="date",
               y="value",
               color="type")) +
    geom_line(aes(size='type')) +
    scale_color_manual(values=color_map) +
    scale_size_manual(values=size_map) +
    scale_x_datetime(labels=date_format('%Y')) +
    theme_bw(base_size=30) +
    theme(legend_position=(0.02, 0.98),
          legend_justification=(0, 1),
          legend_title=element_blank(),
          legend_key=element_rect(fill="none", colour="none"),
          legend_background=element_rect(fill="none", colour="black", alpha=0.3),
          legend_margin=6,
          panel_grid_major_x=element_line(color="grey", alpha=0.05),
          panel_grid_minor_x=element_line(color="grey", alpha=0.05),
          panel_grid_major_y=element_line(color="grey", alpha=0.05),
          panel_grid_minor_y=element_line(color="grey", alpha=0.05)
          ) +
    labs(title="",
         x="",
         y="") +
    guides(
        color=guide_legend(nrow=2, byrow=True),
        size=guide_legend(nrow=2, byrow=True)
    )
)

save_as_pdf_pages(save_plot(trend_plot), filename=f"{plots_dir}/STL_Trend.pdf", verbose=False)
trend_plot[0].save(f"{plots_dir}/STL_Trend.png", dpi=150, verbose=False)
trend_plot[0]

## Seasonality

In [ ]:
seasonality_ht = pd.DataFrame({
    "date": train_df["date"],
    "value": seasonality.reshape(-1,),
    "type": "Hyper-Tree",
})

seasonality_stl = pd.DataFrame({
    "date": train_df["date"],
    "value": res._seasonal.values,
    "type": "STL",
})

seasonality_df = pd.concat([seasonality_ht, seasonality_stl], axis=0)

seasonality_plot = []
seasonality_plot.append(
    ggplot(seasonality_df,
           aes(x="date",
               y="value",
               color="type")) +
    geom_line(aes(size='type')) +
    scale_color_manual(values=color_map) +
    scale_size_manual(values=size_map) +
    scale_x_datetime(labels=date_format('%Y')) +
    theme_bw(base_size=30) +
    theme(legend_position=(0.02, 0.98),
          legend_justification=(0, 1),
          legend_title=element_blank(),
          legend_key=element_rect(fill="none", colour="none"),
          legend_background=element_rect(fill="none", colour="black", alpha=0.3),
          legend_margin=6,
          panel_grid_major_x=element_line(color="grey", alpha=0.05),
          panel_grid_minor_x=element_line(color="grey", alpha=0.05),
          panel_grid_major_y=element_line(color="grey", alpha=0.05),
          panel_grid_minor_y=element_line(color="grey", alpha=0.05)
          ) +
    labs(title="",
         x="",
         y="") +
    guides(
        color=guide_legend(nrow=2, byrow=True),
        size=guide_legend(nrow=2, byrow=True)
    )
)

save_as_pdf_pages(save_plot(seasonality_plot), filename=f"{plots_dir}/STL_Seasonality.pdf", verbose=False)
seasonality_plot[0].save(f"{plots_dir}/STL_Seasonality.png", dpi=150, verbose=False)
seasonality_plot[0]

## STL Parameters

In [ ]:
color_map = {"Hyper-Tree": "#1f77b4"}
size_map = {"Hyper-Tree": 2}


for i in range(ht_stl.n_params):

    param_df = pd.DataFrame({
        "date": train_df["date"],
        "value": model_params.drop(columns="date").iloc[:,i],
        "type": "Hyper-Tree",
    })


    param_plot =[]
    param_plot.append(
        ggplot(param_df,
               aes(x="date",
                   y="value",
                   color="type")) +
        geom_line(aes(size='type')) +
        scale_color_manual(values=color_map) +
        scale_size_manual(values=size_map) +
        scale_x_datetime(labels=date_format('%Y')) +
        theme_bw(base_size=40) +
        theme(legend_position="none",
              panel_grid_major_x=element_line(color="grey", alpha=0.05),
              panel_grid_minor_x=element_line(color="grey", alpha=0.05),
              panel_grid_major_y=element_line(color="grey", alpha=0.05),
              panel_grid_minor_y=element_line(color="grey", alpha=0.05)
              ) +
        labs(title="",
             x="",
             y="") +
        guides(
            color=guide_legend(nrow=2, byrow=True),
            size=guide_legend(nrow=2, byrow=True)
        )
    )

    if i < 2:
        file_name = f"{plots_dir}/STL_a{i}.pdf"
        png_name  = f"{plots_dir}/STL_a{i}.png"
    elif i == 2:
        file_name = f"{plots_dir}/STL_c1.pdf"
        png_name  = f"{plots_dir}/STL_c1.png"
    else:
        file_name = f"{plots_dir}/STL_d1.pdf"
        png_name  = f"{plots_dir}/STL_d1.png"

    print(file_name)

    save_as_pdf_pages(save_plot(param_plot), filename=file_name, verbose = False)
    param_plot[0].save(png_name, dpi=150, verbose=False)

# SHAP Values

In [ ]:
# Initialize SHAP explainer
explainer = shap.TreeExplainer(ht_stl.model)
shap_values = explainer(train_df[features])

param_pos = 0
plt.figure()
shap.plots.bar(shap_values[:, :, param_pos], show=False)
plt.savefig(f"{plots_dir}/shap_a0.pdf", bbox_inches="tight")
plt.savefig(f"{plots_dir}/shap_a0.png", bbox_inches="tight", dpi=150)

param_pos = 1
plt.figure()
shap.plots.bar(shap_values[:, :, param_pos], show=False)
plt.savefig(f"{plots_dir}/shap_a1.pdf", bbox_inches="tight")
plt.savefig(f"{plots_dir}/shap_a1.png", bbox_inches="tight", dpi=150)

param_pos = 2
plt.figure()
shap.plots.bar(shap_values[:, :, param_pos], show=False)
plt.savefig(f"{plots_dir}/shap_c1.pdf", bbox_inches="tight")
plt.savefig(f"{plots_dir}/shap_c1.png", bbox_inches="tight", dpi=150)

param_pos = 3
plt.figure()
shap.plots.bar(shap_values[:, :, param_pos], show=False)
plt.savefig(f"{plots_dir}/shap_d1.pdf", bbox_inches="tight")
plt.savefig(f"{plots_dir}/shap_d1.png", bbox_inches="tight", dpi=150)